In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

In [2]:
df = pd.read_csv("../../data/wustl_iiot_2021.csv")
df.head()

,StartTime,LastTime,SrcAddr,DstAddr,Mean,Sport,Dport,SrcPkts,DstPkts,TotPkts,...,SAppBytes,DAppBytes,TotAppByte,SynAck,RunTime,sTos,SrcJitAct,DstJitAct,Traffic,Target
0,2019-08-19 12:23:28,2019-08-19 12:23:28,192.168.0.20,192.168.0.2,0,59034,502,10,8,18,...,24,20,44,0.001176,0.053037,0,0.000000,0.0,normal,0
1,2019-08-19 15:13:24,2019-08-19 15:13:24,192.168.0.20,192.168.0.2,0,55841,502,10,8,18,...,24,20,44,0.001308,0.052681,0,0.000000,0.0,normal,0
2,2019-08-19 13:41:31,2019-08-19 13:41:31,192.168.0.20,192.168.0.2,0,63774,502,10,8,18,...,24,20,44,0.000690,0.051793,0,0.000000,0.0,normal,0
3,2019-08-19 12:43:19,2019-08-19 12:43:20,209.240.235.92,192.168.0.2,0,61771,80,4,0,4,...,0,0,0,0.000000,0.889555,0,419.338813,0.0,DoS,1
4,2019-08-19 14:49:44,2019-08-19 14:49:48,192.168.0.20,192.168.0.1,3,0,0,14,0,14,...,476,0,476,0.000000,3.500055,0,525.146562,0.0,normal,0


In [41]:
df["StartTime"] = pd.to_datetime(df["StartTime"])
df["LastTime"] = pd.to_datetime(df["LastTime"])
df = df.sort_values("StartTime").reset_index(drop=True)
df["nearest_sec"] = df["StartTime"].dt.floor("s")

In [44]:
per_second = (
    df.groupby(["SrcAddr", "DstAddr", "nearest_sec"])
      .agg(
          flow_count=("StartTime", "size"),

          src_pkts=("SrcPkts", "sum"),
          dst_pkts=("DstPkts", "sum"),
          tot_pkts=("TotPkts", "sum"),

          src_bytes=("SrcBytes", "sum"),
          dst_bytes=("DstBytes", "sum"),
          tot_bytes=("TotBytes", "sum"),

          duration_mean=("Dur", "mean"),

          src_rate_mean=("SrcRate", "mean"),
          dst_rate_mean=("DstRate", "mean"),
          rate_mean=("Rate", "mean"),

          src_load_mean=("SrcLoad", "mean"),
          dst_load_mean=("DstLoad", "mean"),

          src_jitter_mean=("SrcJitter", "mean"),
          dst_jitter_mean=("DstJitter", "mean"),

          src_loss=("SrcLoss", "sum"),
          dst_loss=("DstLoss", "sum"),
          loss=("Loss", "sum"),

          tcp_rtt_mean=("TcpRtt", "mean"),
          traffic = ("Traffic", "first"),
          target=("Target", "max")
      )
      .reset_index()
)

In [40]:
display(per_second)

,SrcAddr,DstAddr,epochTime,flow_count,src_pkts,dst_pkts,tot_pkts,src_bytes,dst_bytes,tot_bytes,...,dst_load_mean,src_jitter_mean,dst_jitter_mean,src_loss,dst_loss,loss,tcp_rtt_mean,traffic,target,window_id
0,0,0,1566207,1,0,0,0,0,0,0,...,0.0,0.000000,0.0,0,0,0,0.0,normal,0,1566180
1,0,0,1566221,1,0,0,0,0,0,0,...,0.0,0.000000,0.0,0,0,0,0.0,normal,0,1566180
2,0,14740,1566212,1,90864,11501,90864,6625600,0,2019535332,...,0.0,0.000000,0.0,0,0,0,0.0,normal,0,1566180
3,0,15046,1566217,1,11560267,154248,11560267,878841774,0,0,...,0.0,0.000000,0.0,0,0,0,0.0,normal,0,1566180
4,0,16274,1566213,1,93115,14011,93115,6761528,0,2138927796,...,0.0,0.000000,0.0,0,0,0,0.0,normal,0,1566180
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
874,fe80::e9ed:931f:c2e0:1333,ff02::1:3,1566228,1,4,0,4,360,0,360,...,0.0,47.114758,0.0,0,0,0,0.0,normal,0,1566180
875,fe80::e9ed:931f:c2e0:1333,ff02::1:3,1566229,1,4,0,4,360,0,360,...,0.0,47.176043,0.0,0,0,0,0.0,normal,0,1566180
876,fe80::e9ed:931f:c2e0:1333,ff02::1:3,1566230,2,8,0,8,720,0,720,...,0.0,46.969805,0.0,0,0,0,0.0,normal,0,1566180
877,fe80::e9ed:931f:c2e0:1333,ff02::1:3,1566231,1,4,0,4,360,0,360,...,0.0,47.100148,0.0,0,0,0,0.0,normal,0,1566180


In [16]:
# to fill nan for std
agg_df = agg_df.fillna(0)
agg_df = agg_df.sort_values(["SrcAddr", "DstAddr", "window"]).reset_index(drop=True)

In [29]:
display(agg_df)

,window_id,SrcAddr,DstAddr,src_pkts_sum,src_pkts_mean,src_pkts_max,src_pkts_std,dst_pkts_sum,dst_pkts_mean,dst_pkts_max,...,src_jit_act_mean,dst_jit_act_mean,unique_sports,unique_dports,unique_protocols,sttl_mean,dttl_mean,stos_mean,synack_mean,target
0,1566180,0,0,0,0.0,0,0.000000,0,0.0,0,...,0.000000,0.0,1,1,1,0.0,0.0,0.0,0.0,0
1,1566180,0,14740,90864,90864.0,90864,NaN,11501,11501.0,11501,...,0.000000,0.0,1,1,1,0.0,0.0,0.0,0.0,0
2,1566180,0,15046,11560267,11560267.0,11560267,NaN,154248,154248.0,154248,...,0.000000,0.0,1,1,1,0.0,0.0,0.0,0.0,0
3,1566180,0,16274,93115,93115.0,93115,NaN,14011,14011.0,14011,...,0.000000,0.0,1,1,1,0.0,0.0,0.0,0.0,0
4,1566180,0,16436,57040,57040.0,57040,NaN,13854,13854.0,13854,...,0.000000,0.0,1,1,1,0.0,0.0,0.0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
146,1566180,fe80::dacb:8aff:fe08:ed2a,ff02::1,24,3.0,4,1.069045,0,0.0,0,...,864.225047,0.0,1,2,1,1.0,0.0,166.5,0.0,0
147,1566180,fe80::dacb:8aff:fe08:ed2a,ff02::2,8,1.0,1,0.000000,0,0.0,0,...,0.000000,0.0,1,1,1,255.0,0.0,0.0,0.0,0
148,1566180,fe80::e9ed:931f:c2e0:1333,ff02::16,270,10.0,10,0.000000,0,0.0,0,...,77.244742,0.0,1,1,1,1.0,0.0,0.0,0.0,0
149,1566180,fe80::e9ed:931f:c2e0:1333,ff02::1:2,840,2.8,6,1.602673,0,0.0,0,...,1359.993418,0.0,1,1,1,1.0,0.0,0.0,0.0,0


In [26]:
exclude_cols = [
    "window",
    "SrcAddr",
    "DstAddr",
    "target"
]

feature_cols = [
    col for col in agg_df.columns
    if col not in exclude_cols
]

X = agg_df[feature_cols]
y = agg_df["target"]

In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [28]:
model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train,
    y_train,
    eval_set=[
        (X_train, y_train),
        (X_val, y_val)
    ],
    verbose=False
)

XGBoostError: [21:14:40] /Users/runner/work/xgboost/xgboost/src/data/quantile_dmatrix.cc:58: Check failed: ref->Info().num_col_ == n_features (60 vs. 59) : Invalid ref DMatrix, different number of features.
Stack trace:
  [bt] (0) 1   libxgboost.dylib                    0x000000012cfe7dd8 dmlc::LogMessageFatal::~LogMessageFatal() + 124
  [bt] (1) 2   libxgboost.dylib                    0x000000012d213f00 xgboost::data::GetCutsFromRef(xgboost::Context const*, std::__1::shared_ptr<xgboost::DMatrix>, unsigned int, xgboost::BatchParam, xgboost::common::HistogramCuts*) + 388
  [bt] (2) 3   libxgboost.dylib                    0x000000012d21998c xgboost::data::cpu_impl::MakeSketches(xgboost::Context const*, xgboost::data::DataIterProxy<void (void*), int (void*)>*, xgboost::data::DMatrixProxy*, std::__1::shared_ptr<xgboost::DMatrix>, float, xgboost::common::HistogramCuts*, xgboost::BatchParam const&, xgboost::MetaInfo const&, xgboost::data::ExternalDataInfo const&, std::__1::vector<xgboost::Fe

In [14]:
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       800

    accuracy                           1.00       800
   macro avg       1.00      1.00      1.00       800
weighted avg       1.00      1.00      1.00       800

[[800]]


/Users/ryan/Documents/NUS/Y4S1/DSA4266/network-traffic-classification/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:614: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
